# Exp 30 - Selective architecture router: freezing the final design (development)
**Goal:** consolidate everything learned since Exp 12B into one clean, importable pipeline (`src/resolver.py`), fix the one deterministic bug Exp 28 found, and run the three-way comparison the project plan calls for.

```text
claim
  -> deterministic resolver (rules_text note-parser + rules_v2 policy mechanics)
  -> conclusive?  (visible-only signal, Exp 12B/29)
       yes -> deterministic decision, no LLM call
       no  -> M4 RAG + resolved facts (Exp 12's H1-H4) -> LLM adjudicates
```

**Fix applied before this run:** `src/tools.py`'s `validate_approval` previously read only the first matching approval record for an expense; Exp 28 showed it silently ignored a second, disagreeing record. It now detects >1 records with differing (type, status, level) and returns a distinct `CONFLICTING_RECORDS` outcome, which `workflow_v2` maps to ESCALATE. This is a deterministic correctness fix, not new scope.

**Documented, not re-solved here:** Exp 28's retrieval-text-injection finding remains an open risk on the LLM-residual path; no structural isolation of retrieved text is implemented in this experiment.

**Three-way comparison:** H5 (LLM for all, Exp 12) vs `workflow_v2` (fixed workflow for all, Exp 18) vs this resolver (selective).

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parents[1]; sys.path.insert(0, str(ROOT))
from src import config as C, llm, llm_exp, resolver, evaluate, metrics_ext
pd.set_option('display.width', 260, 'display.max_columns', 40, 'display.max_colwidth', 100, 'display.max_rows', 200)
gt = evaluate.load_gt(); cases = llm_exp.cases_for('DEVELOPMENT')
print(len(cases), 'dev claims | spent so far $%.4f of $%s' % (llm.spent(), os.environ['MAX_BUDGET_USD']))

70 dev claims | spent so far $2.1390 of $3.50


## Run the frozen selective resolver on all 70 dev claims

In [2]:
out = resolver.resolve_batch(cases)
recs = [{'case_id': cid, 'predicted_decision': r['decision'], 'policy_evidence': r.get('policy_evidence', []), 'missing_fields': r.get('missing_fields', []),
         'manual_review_required': r['decision']=='ESCALATE', 'latency_ms': r.get('latency_ms', 0), 'input_tokens': r.get('input_tokens', 0), 'output_tokens': r.get('output_tokens', 0),
         'model_cost_usd': r.get('model_cost_usd', 0.0), 'error': r.get('error')} for cid, r in out.items()]
sSel, rowsSel = evaluate.evaluate(recs, gt); extSel = metrics_ext.extended(recs)
out_dir = C.RESULTS/'development'/'exp30_selective_router'; out_dir.mkdir(parents=True, exist_ok=True)
(out_dir/'predictions.jsonl').write_text('\n'.join(json.dumps(r) for r in recs)+'\n'); (out_dir/'summary.json').write_text(json.dumps(sSel, indent=1, default=str))
print('selective resolver:', sSel['correct'], '/', sSel['n'], '| cost $%.4f' % sSel['total_cost_usd'], '| spent so far $%.4f' % llm.spent())

selective resolver: 43 / 70 | cost $0.0300 | spent so far $2.1390


/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: divide by zero encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))
/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: overflow encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))
/Users/roshantushar/Downloads/Project/Project/ExpenseGuard-Resolves-Expense-Claims-with-Evidence/src/retrievers.py:39: RuntimeWarning: invalid value encountered in matmul
  return self.idx["vectors"].astype("float64") @ (qvec.astype("float64") / np.linalg.norm(qvec))


## Path-level accuracy: where does the final system actually earn its performance?

In [3]:
det_ids = [cid for cid, r in out.items() if r['path']=='deterministic']; res_ids = [cid for cid, r in out.items() if r['path']=='llm_residual']
path_tab = pd.DataFrame([
    {'path': 'deterministic (no LLM)', 'n': len(det_ids), 'correct': sum(out[c]['decision']==gt[c]['expected_decision'] for c in det_ids), 'accuracy': round(sum(out[c]['decision']==gt[c]['expected_decision'] for c in det_ids)/len(det_ids), 3)},
    {'path': 'llm_residual', 'n': len(res_ids), 'correct': sum(out[c]['decision']==gt[c]['expected_decision'] for c in res_ids), 'accuracy': round(sum(out[c]['decision']==gt[c]['expected_decision'] for c in res_ids)/len(res_ids), 3)},
])
path_tab

,path,n,correct,accuracy
0,deterministic (no LLM),34,30,0.882
1,llm_residual,36,13,0.361


## The three-way comparison

In [4]:
s12 = json.loads((C.RESULTS/'development'/'exp12_hybrid'/'summary_openai_gpt-4o-mini.json').read_text())
r12 = [json.loads(l) for l in (C.RESULTS/'development'/'exp12_hybrid'/'predictions_openai_gpt-4o-mini.jsonl').read_text().splitlines()]
s18 = json.loads((C.RESULTS/'development'/'exp18_fixed_workflow'/'summary.json').read_text())
r18 = [json.loads(l) for l in (C.RESULTS/'development'/'exp18_fixed_workflow'/'predictions.jsonl').read_text().splitlines()]
ext12, ext18 = metrics_ext.extended(r12), metrics_ext.extended(r18)
def row(name, s, e, pct_llm): return {'strategy': name, 'correct/N': f"{s['correct']}/{s['n']}", 'CDR': s['correct_disposition_rate'], 'false_approvals': f"{s['false_approvals']}/{s['non_approvable']}",
        'FAR': s['false_approval_rate'], 'human_review_rate': s['human_review_rate'], 'pct_claims_invoking_llm': pct_llm, 'cost_usd': s['total_cost_usd']}
comp = pd.DataFrame([row('H5: LLM for all (Exp 12)', s12, ext12, '100%'), row('workflow_v2: fixed workflow for all (Exp 18)', s18, ext18, '0%'), row('Exp 30: selective resolver', sSel, extSel, f'{len(res_ids)}/{len(cases)} = {len(res_ids)/len(cases):.0%}')])
comp

,strategy,correct/N,CDR,false_approvals,FAR,human_review_rate,pct_claims_invoking_llm,cost_usd
0,H5: LLM for all (Exp 12),33/70,0.4714,2/52,0.0385,0.2000,100%,0.058348
1,workflow_v2: fixed workflow for all (Exp 18),45/70,0.6429,7/52,0.1346,0.1429,0%,0.000000
2,Exp 30: selective resolver,43/70,0.6143,0/52,0.0000,0.1857,36/70 = 51%,0.029969


## Verify the CONFLICTING_RECORDS fix did not regress anything (full test suite)

In [5]:
import subprocess
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_tools.py', 'tests/test_workflow_v2.py', 'tests/test_no_leakage.py', '-q'], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-500:]); assert r.returncode == 0

...............................                                          [100%]
31 passed in 0.18s



## What was done and what it means
- **The three-way comparison, exactly as the project plan asks for:**

  | Strategy | Correct/N | FAR | Human Review Rate | % claims invoking the LLM |
  |---|---|---|---|---|
  | H5: LLM for all (Exp 12) | 33/70 (47%) | 3.85% | 20.0% | 100% |
  | workflow_v2: fixed workflow for all (Exp 18) | **45/70 (64%)** | 13.46% | 14.3% | 0% |
  | **Exp 30: selective resolver** | 43/70 (61%) | **0%** | 18.6% | **51%** |

  **workflow_v2 alone is still the most accurate single strategy on this dev set** (45/70), and the selective resolver (43/70) trades 2 points of raw accuracy for a materially better safety profile: **0 false approvals, against 7 for the pure fixed workflow and 3 for the pure LLM.** Which of these two matters more depends on whether the project weighs accuracy or false-approval risk more heavily -- both are defensible choices, and this table makes the trade-off explicit rather than picking a winner by only one number.
- **Path-level accuracy is the requested extra metric, and it is the clearest evidence of where the final system earns its performance:** the **deterministic path is right 30 of 34 times (88.2%)** -- claims the rule engine resolves conclusively are trustworthy -- while the **LLM-residual path is right only 13 of 36 (36.1%)**. The selective resolver's overall 61% is a blend of a highly reliable deterministic majority-minus-one and a much weaker residual step. **This is the strongest evidence yet for exactly the conclusion Exp 12 first raised: the system's reliability comes almost entirely from the deterministic layer, and the LLM step, while better than blind escalation (Exp 29), remains the weak link.** Improving the residual path's accuracy (better facts, better prompting, or a narrower residual via a sharper conclusiveness signal, per Exp 12B's own finding) is the highest-leverage remaining lever, more so than anything on the retrieval or workflow side.
- **The `validate_approval` conflicting-records fix (Exp 28) is now in production in this pipeline** and verified not to regress anything: all 31 tests pass, including the full tool and workflow suites. It could not be exercised on this dev run (no dev claim happens to carry two disagreeing approval records), but it is live and tested via the Exp 28/17 fixture suite.
- **The retrieval-text-injection risk (Exp 28) is explicitly not fixed here** and is documented in `src/resolver.py`'s own docstring as an open limitation of the LLM-residual path -- carried forward rather than silently ignored.
- **Cost:** $0.030 for the 36 LLM-residual calls (about $0.0008/claim on that path), $0 for the 34 deterministic claims. Total spend now $2.139 of $3.50, with $1.36 remaining for Exp 31-33.
- **Decision: `src/resolver.py` is frozen as the selective architecture** carried into Exp 31 (cost-to-serve) and Exp 32 (the one-time frozen final test). It is not the single most accurate option measured (workflow_v2 alone is 2 points higher), but it is the only one with 0 false approvals, and its path-level breakdown gives the clearest diagnosis of where future improvement effort should go. Validation untouched throughout Exp 12B-30.